In [1]:
import os, glob, subprocess, sys, time
import pandas as pd

# ---- locate inputs ----
CODE = "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"
DATA = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

ck = sorted(glob.glob("/kaggle/input/**/fold*.pt", recursive=True))
print("checkpoints found:", len(ck))
for c in ck: print("  ", c)
assert len(ck) == 5, "expected 5 fold*.pt files - is the knee-models dataset attached?"
MODELS = os.path.dirname(ck[0])
assert all(os.path.dirname(c) == MODELS for c in ck), "checkpoints are in different folders"

assert os.path.exists(f"{CODE}/infer.py"), "knee-mri-pipeline dataset not attached / path differs"
for f in ("test.csv", "test_series.csv", "sample_submission.csv"):
    assert os.path.exists(f"{DATA}/{f}"), f"{f} missing in {DATA}"

# ---- run inference ----
t0 = time.time()
cmd = [sys.executable, f"{CODE}/infer.py", "--models_dir", MODELS, "--data_dir", DATA,
       "--D", "24", "--size", "160", "--workers", "4", "--bs", "4",
       "--out", "/kaggle/working/submission.csv"]
r = subprocess.run(cmd)
assert r.returncode == 0, "infer.py failed - see the traceback above"
print(f"inference took {(time.time() - t0) / 60:.1f} min")

# ---- sanity-check the output ----
sub = pd.read_csv("/kaggle/working/submission.csv")
ref = pd.read_csv(f"{DATA}/sample_submission.csv")
assert list(sub.columns) == list(ref.columns), "column mismatch"
assert len(sub) == len(ref) and sub.StudyInstanceUID.tolist() == ref.StudyInstanceUID.tolist(), "row mismatch"
assert sub.drop(columns="StudyInstanceUID").notna().all().all(), "NaNs in submission"
print(sub.shape, "OK")
print(sub.head(3))

checkpoints found: 5
   /kaggle/input/datasets/theokorir/knee-models/knee_models/fold0.pt
   /kaggle/input/datasets/theokorir/knee-models/knee_models/fold1.pt
   /kaggle/input/datasets/theokorir/knee-models/knee_models/fold2.pt
   /kaggle/input/datasets/theokorir/knee-models/knee_models/fold3.pt
   /kaggle/input/datasets/theokorir/knee-models/knee_models/fold4.pt


/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fuse = nn.TransformerEncoder(layer, n_layers)


3/3 | 45s
saved /kaggle/working/submission.csv (3, 13) | unreadable studies: 0 | 45s
inference took 1.2 min
(3, 13) OK
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.466667  0.600000   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.933333  0.866667   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.600000  0.533333   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.533333          0.466667   0.600000    0.733333  0.733333   
1         0.600000          0.666667   0.466667    0.333333  0.333333   
2         0.866667          0.866667   0.933333    0.933333  0.933333   

   Effusion  Synovitis  Baker's  Contusion  Fracture  
0  0.666667        0.6      0.8   0.666667  0.533333  
1  0.733333        0.6      0.4   0.866667  0.933333  
2  0.600000        0.8      0.8   0.466667  0.533333  


In [2]:
import pydicom, pandas as pd, random
from pathlib import Path
from collections import Counter

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
for split in ["train", "test"]:
    root = DATA / f"{split}_series"
    ts = pd.read_csv(DATA / f"{split}_series.csv").sample(300 if split == "train" else 10**6, random_state=0, replace=False) if split == "train" else pd.read_csv(DATA / f"{split}_series.csv")
    ts_counts, bad, tried = Counter(), 0, 0
    for s, e in zip(ts.StudyInstanceUID, ts.SeriesInstanceUID):
        d = root / s / e
        if not d.exists(): continue
        f = next(iter(d.iterdir()), None)
        if f is None: continue
        ds = pydicom.dcmread(f, force=True)          # full read, one slice per series
        ts_counts[str(ds.file_meta.get("TransferSyntaxUID", "none"))] += 1
        try:
            _ = ds.pixel_array; tried += 1
        except Exception as ex:
            bad += 1
            if bad <= 3: print(split, "decode fail:", type(ex).__name__, str(ex)[:150])
    print(split, dict(ts_counts), "| decoded ok:", tried, "| failed:", bad)

train {'1.2.840.10008.1.2.1': 300} | decoded ok: 300 | failed: 0
test {'1.2.840.10008.1.2.1': 15} | decoded ok: 15 | failed: 0
